# Ingest the annual reports into **Qdrant** (text + ColPali visual index)

Code: [scripts/rag/financial_ingest_qdrant_db.py](../scripts/rag/financial_ingest_qdrant_db.py), driven by
[scripts/rag/colab_pipeline.py](../scripts/rag/colab_pipeline.py) (pipeline `colpali_qdrant`). Two collections on a Qdrant
server running on this VM:

| Collection | What | Vectors | RAM / disk |
|---|---|---|---|
| `annual_reports_text` | every page's text: the text layer, **OCR** for garbled / outlined / scanned pages, tables as Markdown | `dense` MiniLM 384-d + `bm25` sparse (number-aware terms, IDF) | dense in RAM (HNSW); payloads on disk |
| `annual_reports_visual` | **ColPali** page images of table, chart and image pages, plus the page's text | `colpali` full 1,030 x 128 fp16 | **disk**, no HNSW (MaxSim rerank only) |
| | | `mean_rows` / `mean_cols`: 32 row / column means + 6 special tokens = 38 x 128 | **RAM**, HNSW (first stage) |
| | | `summary`: MiniLM of the page text | RAM |

```
Drive PDFs -> copy one sector to local disk -> analyse pages (parallel) -> OCR unreadable pages (CPU)
   -> text stage:   chunks -> dense + bm25 -> annual_reports_text
   -> visual stage: page images (1.5x) -> ColPali (GPU) -> pooled + full multivectors -> annual_reports_visual
   -> snapshot both collections + ingest state to Drive after every sector
```

There is **no VLM** in this pipeline: ColPali only finds pages; the text the LLM reads comes from the text layer or OCR.
Search is hybrid (dense + BM25, RRF) and, with `RAG_FIN_VISUAL=1`, adds ColPali pages (pooled prefetch, exact MaxSim rerank).
If Track B later provides Qwen3-VL as the chat model, the pages ColPali finds can be sent to it as images.

**Runtime:** GPU for ColPali (T4 is fine, L4 faster). Rough cost for min500, **estimates**: text stage + page analysis
~3 CPU-hours, OCR of ~8k unreadable pages ~10 CPU-hours, ColPali on ~10k table/chart/image pages a few GPU-hours.
Storage: ~0.26 MB per visual page on disk (full vectors), ~20 KB in RAM (pooled). Snapshots copy whole collections to
Drive, so use `SNAPSHOT_EVERY = 0` (once per sector) at full scale.

**Before you start:** on your PC run `python -m scripts.common.colab_bundle` and upload `output/colab/capstone_code.zip`
to `MyDrive/capstone_rag/`.

In [ ]:
# @title 1. Settings
CORPUS = "min500"  # @param ["min500", "full"]
# min500 = 509 filings (Track B minimum: 41 per sector + every eval-set company); full = all 1,004
SECTORS = ""  # @param {type:"string"}
# "" = all 13 sectors; or e.g. "banking,retail"
MAX_PER_SECTOR = 0  # @param {type:"integer"}
# download only: 0 = every PDF of the CORPUS preset; 20 = a first pass (largest companies first)
SMOKE_TEST = True  # @param {type:"boolean"}
# True: 2 PDFs per sector downloaded, 1 filing per sector ingested, first 30 pages, into a separate *_smoke index
SEC_USER_AGENT = "capstone-research your.name@example.com"  # @param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/capstone_rag"
REPO = "https://github.com/rama159065/Capstone.git"   # used only when there is no code zip
BRANCH = "financial-report-only"
CODE_ZIP = "/content/drive/MyDrive/capstone_rag/capstone_code.zip"   # python -m scripts.common.colab_bundle, then upload
STAGES = "text,visual"  # @param ["text,visual", "text", "visual"]
# text = annual_reports_text (CPU); visual = ColPali into annual_reports_visual (GPU)
COLPALI_PAGES = "tables_charts_images"  # @param ["tables_charts_images", "tables_charts", "visual", "all"]
# pages that get ColPali vectors; every page's text goes to annual_reports_text either way
OCR = "garbled"  # @param ["garbled", "none"]
# garbled = OCR every unreadable page (garbled text layer, outlined glyphs, scans) so its text is searchable
OCR_MAX_PAGES_PER_FILING = 0  # @param {type:"integer"}
# 0 = every unreadable page; e.g. 20 caps the OCR time per filing
SNAPSHOT_EVERY = 0  # @param {type:"integer"}
# batches of 10 filings between Drive snapshots; 0 = once per sector (each snapshot copies the whole collection)
WORKERS = 2  # @param {type:"integer"}
# processes for page analysis (Colab standard runtimes have 2 CPUs)

In [ ]:
# @title 2. Drive, code, dependencies
import json, os, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")


def fetch_code(root, repo_url, branch):
    # Unpack the code zip (Drive, else one you upload) into root; without a zip, clone the repo.
    import zipfile
    uploaded = Path("/content") / Path(CODE_ZIP).name
    source = next((p for p in (Path(CODE_ZIP), uploaded) if p.exists()), None)
    if source is None and not root.exists():
        from google.colab import files
        print(f"{CODE_ZIP} not found: upload capstone_code.zip (cancel to clone {repo_url} instead)")
        names = list(files.upload())
        source = Path("/content") / names[0] if names else None
    if source is not None:
        with zipfile.ZipFile(source) as archive:
            info = archive.read("BUNDLE_INFO.json")
            stamp = root / "BUNDLE_INFO.json"
            if not stamp.exists() or stamp.read_bytes() != info:      # a new build: unpack over the old one
                archive.extractall(root)
        print("code:", source, json.loads(info))
    elif not root.exists():
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")      # needed while the repo is private
        except Exception:
            token = None
        url = repo_url.replace("https://", f"https://{token}@") if token else repo_url
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", branch, url, str(root)], check=True)


fetch_code(Path("/content/Capstone"), REPO, BRANCH)
sys.path.insert(0, "/content/Capstone")
%cd /content/Capstone
CONSTRAINTS = "/content/Capstone/deploy/colab_constraints.txt"   # versions tested on the PC (from uv.lock)
PIN = f"-c {CONSTRAINTS}" if Path(CONSTRAINTS).exists() else ""
!pip -q install {PIN} "qdrant-client>=1.12,<2" "pymupdf>=1.24,<2" "transformers>=4.57,<6" "sentence-transformers>=3" "rapidocr>=3,<4" onnxruntime
import importlib.util
assert importlib.util.find_spec("scripts.rag.financial_ingest_qdrant_db"), \
    "old code: rebuild capstone_code.zip (python -m scripts.common.colab_bundle) and upload it to Drive"
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "no GPU: the text stage works; ColPali will be very slow (use STAGES = 'text')"

In [ ]:
# @title 3. Settings object, Qdrant server, restore the snapshots from Drive
from scripts.rag import colab_pipeline as cp
os.environ["SEC_USER_AGENT"] = SEC_USER_AGENT
s = cp.Settings(
    pipeline="colpali_qdrant", corpus_preset=CORPUS, drive_root=DRIVE_ROOT, sectors=SECTORS, user_agent=SEC_USER_AGENT,
    max_per_sector=2 if SMOKE_TEST else (MAX_PER_SECTOR or None),
    stages=STAGES, visual_pages=COLPALI_PAGES, ocr=OCR, ocr_max_pages_per_filing=OCR_MAX_PAGES_PER_FILING,
    snapshot_every=SNAPSHOT_EVERY, workers=WORKERS,
    run_name="_smoke" if SMOKE_TEST else "",
    qdrant_url=None if SMOKE_TEST else "http://127.0.0.1:6333",      # smoke runs use embedded Qdrant, off the server
    limit_filings=1 if SMOKE_TEST else None,
    max_pages_per_pdf=30 if SMOKE_TEST else None)
cp.prepare(s)          # starts the Qdrant server on this VM (not for smoke runs)
cp.restore(s)          # first session: nothing to restore; later sessions: snapshots from Drive

In [ ]:
# @title 4. Download the chosen sectors to Drive (resumable; skips PDFs already there)
cp.download(s)
!du -sh {DRIVE_ROOT}/annual_reports/*/ 2>/dev/null | sort -k2

In [ ]:
# @title 5. Ingest into Qdrant, one sector at a time (rerun after a disconnect: cells 2-3, then this one)
# Each sector's PDFs are copied to local disk first; snapshots go to Drive after each sector.
results = cp.ingest(s)
results

In [ ]:
# @title 6. Status: points per collection and sector, failures, what is in RAM and on disk
report = cp.status(s)

In [ ]:
# @title 6b. Coverage of the plan: planned -> downloaded -> ingested, per sector
cov = cp.coverage(s)

In [ ]:
# @title 7. Retrieval checks: hybrid text search, then ColPali (pooled prefetch -> full MaxSim rerank)
cp.query(s, "net interest income", sector="banking")
cp.query(s, "net oil-equivalent production", sector="oil_and_gas")
if "visual" in STAGES:
    cp.query(s, "revenue by segment bar chart", sector="banking", visual=True)

## Serve this index: FastAPI + Streamlit on this runtime

```
browser -> Colab port proxy (only you) -> Streamlit :8501 -> FastAPI :8000 -> Qdrant (this VM)
```

`cp.serve(s)` starts both with `cp.serving_env(s)`: `RAG_BACKEND=financial_qdrant`, `RAG_INGEST_DIR=/content/rag/indexes/qdrant_db`, `QDRANT_URL` (serving needs the server, so `SMOKE_TEST = False`). Requests can filter by `ticker`, `year` and `sector`.
For the Track B model add the Colab secrets `LLM_API_BASE`, `LLM_API_KEY`, `LLM_MODEL` (e.g. `Qwen/Qwen3-8B`) and,
for a self-signed endpoint, `LLM_CA_CERT` (path to the CA certificate on Drive). Without them answers use the mock.
Add the secret `LANGSMITH_API_KEY` to trace every request to LangSmith (project `financial-report-copilot-colab`); see docs/observability_langsmith.md.

In [ ]:
# @title 8. Serving dependencies (from pyproject.toml; keeps Colab's CUDA torch) and LLM secrets
import re, tomllib
deps = tomllib.load(open("/content/Capstone/pyproject.toml", "rb"))["project"]["dependencies"]
deps = [d.split("#")[0].strip() for d in deps if re.split(r"[<>=!~ \[]", d)[0] not in {"torch", "torchvision", "jupyterlab"}]
pin = ["-c", "/content/Capstone/deploy/colab_constraints.txt"] if Path("/content/Capstone/deploy/colab_constraints.txt").exists() else []
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pin, *deps])
from google.colab import userdata
for key in ("LLM_API_BASE", "LLM_API_KEY", "LLM_MODEL", "LLM_CA_CERT"):
    try:
        os.environ[key] = userdata.get(key)
    except Exception:
        print(f"{key} is not a Colab secret" + (": answers use llm_provider=mock" if key == "LLM_MODEL" else ""))
try:                                   # optional: LangSmith traces of every request (Colab secret LANGSMITH_API_KEY)
    os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
    os.environ.setdefault("LANGSMITH_TRACING", "true")
    os.environ.setdefault("LANGSMITH_PROJECT", "financial-report-copilot-colab")
    print("LangSmith tracing on, project", os.environ["LANGSMITH_PROJECT"])
except Exception:
    print("LANGSMITH_API_KEY is not a Colab secret: no LangSmith traces")
# os.environ["RAG_FIN_VISUAL"] = "1"     # add ColPali visual hits to answers (loads ColPali on this GPU)

In [ ]:
# @title 9. Start the API and the chat UI
api, ui = cp.serve(s)
import time; time.sleep(8)
from google.colab.output import eval_js
print("Open the chat UI:", eval_js("google.colab.kernel.proxyPort(8501)"))

In [ ]:
# @title 10. Check the API without the UI: sector-filtered hybrid retrieval and one answer
import urllib.request
def post(path, body):
    request = urllib.request.Request(f"http://127.0.0.1:8000{path}", data=json.dumps(body).encode(),
                                     headers={"Content-Type": "application/json"})
    return json.load(urllib.request.urlopen(request, timeout=300))
hits = post("/retrieve/financial_report_qa", {"query": "net interest income", "sector": "banking", "top_k": 3})
for e in hits["evidence"]:
    print(e["chunk_id"], e.get("retrieval_ranks"), (e.get("preview") or "")[:80])
provider = "hosted" if os.getenv("LLM_MODEL") else "mock"
answer = post("/chat/financial_report_qa", {"query": "What was JPMorgan's net interest income?", "ticker": "JPM",
                                            "year": 2024, "llm_provider": provider})
print(provider, "|", answer["final_status"], "|", (answer.get("answer") or "")[:300])

In [ ]:
# @title 11. Stop serving (the index stays)
for process in (ui, api):
    process.terminate()

In [ ]:
# @title 12. Optional: copy the Drive index copies to a Google Cloud Storage bucket (for the GCP VM)
# The VM then runs `sh deploy/sync_from_gcs.sh` (docs/deployment.md section 7). Needs a GCP project with billing.
GCS_BUCKET = ""                     # @param {type:"string"}   e.g. gs://capstone-rag-yourname
GCP_PROJECT = ""                    # @param {type:"string"}
if GCS_BUCKET:
    from google.colab import auth
    auth.authenticate_user()
    if GCP_PROJECT:
        !gcloud config set project {GCP_PROJECT}
    !gcloud storage buckets describe {GCS_BUCKET} >/dev/null 2>&1 || gcloud storage buckets create {GCS_BUCKET} --location=us-central1
    cp.export_to_gcs(s, GCS_BUCKET)
else:
    print("set GCS_BUCKET to export; skipped")

## Next

- **Real run:** set `SMOKE_TEST = False` and run cells 3-6. After a disconnect run cells 2-3 (they restart Qdrant and
  restore the snapshots) and then 5: finished filings and stages are skipped.
- **Text first, ColPali later:** run with `STAGES = "text"` on a CPU runtime, then `STAGES = "visual"` on a GPU runtime.
- **Storage:** the Drive snapshots of the full plan need more than the free 15 GB (about 17-29 GB of ColPali vectors for
  the full corpus; min500 is about half).
- **Serving on the Docker server:** copy `MyDrive/capstone_rag/snapshots/qdrant_db` into `./snapshots/qdrant_db`, then
  `docker compose run --rm --user root api sh deploy/restore_index.sh` (Qdrant restore + ingest state into
  `<RAG_HOME>/indexes/qdrant_db`).
- **Deploying on GCP:** cell 12 copies the Drive copies to a bucket; on the VM `sh deploy/sync_from_gcs.sh` pulls and
  restores them (see `docs/deployment.md` section 7).
- **Sectors:** banking, insurance, financial_services, oil_and_gas, energy_and_utilities, healthcare, technology_and_it, retail, telecom_and_media, industrials, consumer_goods, materials_and_chemicals, real_estate.